# Lab 5.1: Training CNNs for Remote Sensing Classification

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates training a Convolutional Neural Network (CNN) for remote sensing land cover classification using PyTorch Lightning. We focus on handling **class imbalance** through different sampling strategies.

---

## Key Learning Objectives

By the end of this lab, you will understand:

1. **Class Imbalance**: Why most remote sensing datasets have imbalanced classes and how to detect it
2. **Sampling Strategies**: Comparison of random sampling vs. weighted sampling approaches
3. **CNN Architecture**: A practical, beginner-friendly convolutional model for patch-based classification
4. **Training & Validation**: Using PyTorch Lightning for simplified training workflows
5. **Evaluation Metrics**: Looking beyond accuracy to balanced accuracy and per-class metrics

---

## Project Context

**Inputs (From Lab 3.1):**
- Sentinel-2 patches (4 spectral bands)
- CORINE labels
- NPZ format: `combined_training_data.npz`

**What You'll Do:**
- Analyze class distribution in your dataset
- Compare strategies to handle imbalance: random sampling vs. class weighting
- Train a CNN classifier on batches
- Evaluate performance beyond simple accuracy

**Output:**
- Trained model weights
- Training metrics and validation performance
- Understanding of which strategy works better for your data

## Part 1: Setup and Dependencies

First, import the required libraries.

In [1]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

## Part 2: Define the CNN Model

We'll build a simple Convolutional Neural Network suitable for 3×3 spectral patches. The CNN uses spatial convolutions to learn local patterns directly from the data.

## Part 3: Custom Dataset Class

Create a PyTorch Dataset for loading remote sensing data.

In [2]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.

    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

## Part 4: Data Loading and Preprocessing

Load the training and validation datasets from CSV files.

In [3]:
import numpy as np

# --- 1) Load from your .npz ---
npz_path = "/p/scratch/training2600/kristinsson3/training_data/combined_training_data.npz"
data = np.load(npz_path)

print("Keys in npz:", list(data.keys()))

if "patches" in data and "labels" in data:
    X = data["patches"]
    y = data["labels"]
else:
    possible_X_keys = ["X", "acquisitions", "inputs", "features", "patch"]
    possible_y_keys = ["y", "targets", "labels", "corine", "target"]

    X_key = next((k for k in possible_X_keys if k in data), None)
    y_key = next((k for k in possible_y_keys if k in data), None)

    if X_key is None or y_key is None:
        raise KeyError(
            f"Could not find (patches, labels) in {npz_path}. "
            f"Available keys: {list(data.keys())}"
        )

    X = data[X_key]
    y = data[y_key]

print(f"Loaded X shape: {X.shape}, dtype={X.dtype}")
print(f"Loaded y shape: {y.shape}, dtype={y.dtype}")

# --- 2) Preprocess ---
X = X.astype(np.float32) * 0.0001
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")
n_features = X_flat.shape[1]  # e.g. 36 for 3x3x4, or 90 for 3x3x10

# --- 3) Remap labels to contiguous 0..K-1 ---
# CORINE codes are sparse (1, 2, 3, 11, 12, ..., 25).
# Without remapping, num_classes = 26 but only 10 are real,
# leaving 16 ghost output neurons that confuse training.
unique_labels = np.unique(y)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y = np.array([label_to_idx[int(lbl)] for lbl in y], dtype=np.int64)

print(f"\nLabel remapping (original -> index):")
for orig, idx in label_to_idx.items():
    print(f"  CORINE {orig:2d} -> class index {idx}")
print(f"Total classes: {len(unique_labels)}")


# --- 4) Split into train / val / test ---
def make_splits_train_val_test(
    X,
    y,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
    stratify=True,
):
    """
    Train/Val/Test split with optional stratification.

    Requirements:
      train_frac + val_frac + test_frac == 1.0
    """
    total = train_frac + val_frac + test_frac
    assert abs(total - 1.0) < 1e-6, "train_frac + val_frac + test_frac must be 1.0"

    rng = np.random.default_rng(seed)
    n = len(y)

    if not stratify:
        idx = rng.permutation(n)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)

        train_idx = idx[:n_train]
        val_idx = idx[n_train : n_train + n_val]
        test_idx = idx[n_train + n_val :]
        return (
            (X[train_idx], y[train_idx]),
            (X[val_idx], y[val_idx]),
            (X[test_idx], y[test_idx]),
        )

    # Stratified: do per-class splitting
    classes, y_inv = np.unique(y, return_inverse=True)

    train_idx_all, val_idx_all, test_idx_all = [], [], []

    for c in range(len(classes)):
        cls_idx = np.where(y_inv == c)[0]
        cls_idx = rng.permutation(cls_idx)

        n_c = len(cls_idx)
        n_train_c = int(n_c * train_frac)
        n_val_c = int(n_c * val_frac)

        train_idx_all.append(cls_idx[:n_train_c])
        val_idx_all.append(cls_idx[n_train_c : n_train_c + n_val_c])
        test_idx_all.append(cls_idx[n_train_c + n_val_c :])

    train_idx = rng.permutation(np.concatenate(train_idx_all))
    val_idx = rng.permutation(np.concatenate(val_idx_all))
    test_idx = rng.permutation(np.concatenate(test_idx_all))

    return (
        (X[train_idx], y[train_idx]),
        (X[val_idx], y[val_idx]),
        (X[test_idx], y[test_idx]),
    )


(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_flat,
    y,
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    seed=42,
    stratify=True,
)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features/sample:    {X_train.shape[1]}")


# --- 5) Sanity check label distribution ---
def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        orig = idx_to_label[int(v)]
        print(f"  idx {int(v):2d} (CORINE {orig:2d}): {int(c)}")


label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Keys in npz: ['patches', 'labels']
Loaded X shape: (200000, 3, 3, 4), dtype=float32
Loaded y shape: (200000,), dtype=uint8
Flattened X shape: (200000, 36)

Label remapping (original -> index):
  CORINE  6 -> class index 0
  CORINE  7 -> class index 1
  CORINE 20 -> class index 2
  CORINE 21 -> class index 3
  CORINE 24 -> class index 4
  CORINE 25 -> class index 5
  CORINE 29 -> class index 6
  CORINE 36 -> class index 7
  CORINE 40 -> class index 8
  CORINE 41 -> class index 9
Total classes: 10

Training samples:   159997
Validation samples: 19997
Test samples:       20006
Features/sample:    36

Train label distribution (top 10):
  idx  4 (CORINE 24): 109856
  idx  6 (CORINE 29): 23040
  idx  7 (CORINE 36): 11571
  idx  9 (CORINE 41): 6736
  idx  5 (CORINE 25): 5795
  idx  3 (CORINE 21): 1318
  idx  2 (CORINE 20): 569
  idx  0 (CORINE  6): 419
  idx  8 (CORINE 40): 412
  idx  1 (CORINE  7): 281

Val label distribution (top 10):
  idx  4 (CORINE 24): 13732
  idx  6 (CORINE 29): 2880
 

## Part 5: Understanding Class Imbalance & Sampling Strategies

Before training, we need to understand our data distribution and choose a strategy to handle class imbalance.

In [4]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

print("=" * 70)
print("TRAINING SET CLASS DISTRIBUTION")
print("=" * 70)
print(f"Total samples: {len(y_train_np)}")
print(f"Number of classes: {len(classes)}\n")

# Sort by count (descending)
sort_idx = np.argsort(-counts)
print("Classes (sorted by frequency):")
print("Index | CORINE | Count  | Percentage | Balance")
print("-" * 50)
for idx in sort_idx:
    c, n = classes[idx], counts[idx]
    pct = 100 * n / len(y_train_np)
    bar = "█" * int(pct / 2)
    print(
        f"{int(c):5d} | {idx_to_label[int(c)]:6d} | {int(n):6d} | {pct:6.1f}%  | {bar}"
    )

majority_class_pct = 100 * counts.max() / len(y_train_np)
print(f"\n⚠️  Majority class represents {majority_class_pct:.1f}% of data")
print(f"    This is the baseline accuracy for a 'dumb' classifier!")

# Compute imbalance ratio
imbalance_ratio = counts.max() / counts.min()
print(f"    Imbalance ratio (max/min): {imbalance_ratio:.1f}x")

TRAINING SET CLASS DISTRIBUTION
Total samples: 159997
Number of classes: 10

Classes (sorted by frequency):
Index | CORINE | Count  | Percentage | Balance
--------------------------------------------------
    4 |     24 | 109856 |   68.7%  | ██████████████████████████████████
    6 |     29 |  23040 |   14.4%  | ███████
    7 |     36 |  11571 |    7.2%  | ███
    9 |     41 |   6736 |    4.2%  | ██
    5 |     25 |   5795 |    3.6%  | █
    3 |     21 |   1318 |    0.8%  | 
    2 |     20 |    569 |    0.4%  | 
    0 |      6 |    419 |    0.3%  | 
    8 |     40 |    412 |    0.3%  | 
    1 |      7 |    281 |    0.2%  | 

⚠️  Majority class represents 68.7% of data
    This is the baseline accuracy for a 'dumb' classifier!
    Imbalance ratio (max/min): 390.9x


## Part 6: Setting up the Model

In [5]:
import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvNet(pl.LightningModule):
    """
    Simple CNN classifier for square Sentinel-2 patches.

    Assumes flat input (B, h*w*in_channels) where spatial layout is
    channels-last: the flat was produced by X.reshape(N, -1) on an
    array of shape (N, h, w, in_channels).

    patch_size: spatial size (h = w), default 3 for 3x3 patches.
    in_channels: number of spectral bands (4 or 10).
    """

    def __init__(
        self,
        num_classes=10,
        in_channels=10,
        patch_size=3,
        lr=3e-4,
        max_epochs=60,
        class_weights=None,
    ):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        """
        Convert flat input to image format (B, C, H, W).
        Handles channels-last layout: (B, h*w*c) → (B, h, w, c) → (B, c, h, w)
        """
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            assert (
                x.size(1) == h * w * c
            ), f"Expected {h * w * c} features, got {x.size(1)}"
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        x = self._to_image(x)
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

In [6]:
# ============================================================================
# CONFIGURATION: Choose Your Sampling Strategy
# ============================================================================

batch_size = 256  # Reduced for better class mixing with sampling
max_epochs = 100

print("\n" + "=" * 70)
print("SAMPLING STRATEGY COMPARISON")
print("=" * 70)

# Strategy 1: Inverse-frequency weights for weighted loss
print("\n1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each class gets a weight = 1 / frequency")
print("   • Model sees all samples in their original distribution")
print("   • Loss contribution per class is equalized")
print("   • Pros: Simple, no data duplication")
print("   • Cons: Model still biased toward majority class")

# Compute sqrt-scaled weights (softens extreme weights)
raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n

sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)  # Normalize to mean=1

print("\n   Class weights (sqrt-scaled):")
print("   Index | CORINE | Weight")
print("   " + "-" * 35)
sort_idx = np.argsort(-sqrt_weights)
for c_idx in sort_idx[: len(classes)]:
    orig = idx_to_label[c_idx]
    print(f"   {c_idx:5d} | {orig:6d} | {sqrt_weights[c_idx]:6.2f}")

class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Strategy 2: Weighted Random Sampler
print("\n\n2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)")
print("-" * 70)
print("   How it works:")
print("   • Each sample gets weight = 1 / class_frequency")
print("   • Sampler draws batches with replacement")
print("   • Rare classes appear more often in mini-batches")
print("   • Pros: Direct oversampling, minorities get more epochs")
print("   • Cons: Duplication, may overfit on rare classes")

# Map each sample to its weight
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}
sample_weights = np.array(
    [class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64
)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("   ✓ Sampler created")

# ============================================================================
# SELECT WHICH STRATEGY TO USE
# ============================================================================
print("\n\n" + "=" * 70)
print("YOUR CHOICE: Which strategy will you use?")
print("=" * 70)

strategy = "weighted_sampler"  # ← CHANGE THIS TO "weighted_sampler" TO COMPARE

if strategy == "class_weights":
    print(f"\n✅ Selected: CLASS-WEIGHTED LOSS")
    class_weights_for_loss = class_weights_t
    train_sampler = None
    train_shuffle = True
    print(f"   • Using loss weights to balance classes")
    print(f"   • DataLoader will shuffle normally")

elif strategy == "weighted_sampler":
    print(f"\n✅ Selected: WEIGHTED RANDOM SAMPLER")
    class_weights_for_loss = None
    train_sampler = sampler
    train_shuffle = False
    print(f"   • Using resampling in DataLoader")
    print(f"   • Rare classes will appear more often")

else:
    print(f"\n✅ Selected: NO WEIGHTS")
    class_weights_for_loss = None
    train_sampler = None
    train_shuffle = True
    print(f"   • DataLoader will shuffle normally")

print("\n" + "=" * 70)
print("MODEL SETUP")
print("=" * 70)

# Auto-detect architecture parameters
num_classes = len(np.unique(y_train_np))
patch_size = 3  # 3x3 patches
in_channels = X_train.shape[1] // (patch_size**2)

print(f"\n  num_classes    = {num_classes}")
print(f"  patch_size     = {patch_size}×{patch_size}")
print(f"  in_channels    = {in_channels}")
print(f"  batch_size     = {batch_size}")

# Create model
model = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_for_loss,
    max_epochs=max_epochs,
)

print(f"\n✓ CNN Model created")

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

# Create dataloaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"✓ DataLoaders created")
print(f"  • Training batches:   {len(train_dataloader)}")
print(f"  • Validation batches: {len(val_dataloader)}")
print(f"  • Test batches:       {len(test_dataloader)}")


SAMPLING STRATEGY COMPARISON

1️⃣  STRATEGY 1: Class-Weighted Loss (No Resampling)
----------------------------------------------------------------------
   How it works:
   • Each class gets a weight = 1 / frequency
   • Model sees all samples in their original distribution
   • Loss contribution per class is equalized
   • Pros: Simple, no data duplication
   • Cons: Model still biased toward majority class

   Class weights (sqrt-scaled):
   Index | CORINE | Weight
   -----------------------------------
       1 |      7 |   2.20
       8 |     40 |   1.81
       0 |      6 |   1.80
       2 |     20 |   1.54
       3 |     21 |   1.01
       5 |     25 |   0.48
       9 |     41 |   0.45
       7 |     36 |   0.34
       6 |     29 |   0.24
       4 |     24 |   0.11


2️⃣  STRATEGY 2: Weighted Random Sampler (Resampling)
----------------------------------------------------------------------
   How it works:
   • Each sample gets weight = 1 / class_frequency
   • Sampler draws bat

## Part 7: Train Your Model

Now let's train the CNN with your chosen sampling strategy. Monitor the training and validation metrics to see how imbalance affects learning.

In [7]:
# Verify data shapes before training
xb, yb = next(iter(train_dataloader))
print("Quick data check:")
print(f"  Batch X shape: {xb.shape} (dtype={xb.dtype})")
print(f"  Batch y shape: {yb.shape} (dtype={yb.dtype})")
print(f"  y range:       [{yb.min().item()}, {yb.max().item()}]")
print(f"  Unique classes in batch: {sorted(torch.unique(yb).tolist())}")

Quick data check:
  Batch X shape: torch.Size([256, 36]) (dtype=torch.float32)
  Batch y shape: torch.Size([256]) (dtype=torch.int64)
  y range:       [0, 9]
  Unique classes in batch: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [8]:
print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Strategy:     {strategy}")
print(f"Max epochs:   {max_epochs}")
print(f"Batch size:   {batch_size}")
print("=" * 70 + "\n")

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=max_epochs,
    gradient_clip_val=1.0,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)

print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)
trainer.test(model, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/p/project1/training2600/kristinsson3/envs/ml_eo_course/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enab


STARTING TRAINING
Strategy:     weighted_sampler
Max epochs:   100
Batch size:   256



┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.9 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.



TESTING


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4010796844959259     │
│         test_loss         │    1.3873374462127686     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.3873374462127686, 'test_acc': 0.4010796844959259}]

### Imbalance-aware validation diagnostics

Accuracy alone can be misleading on imbalanced labels. This cell reports:
- majority-class baseline
- balanced accuracy (mean recall across classes)
- macro-F1 (equal weight per class)
- per-class recall

In [9]:
import numpy as np
import torch


def _collect_preds_targets(model, dataloader):
    model.eval()
    device = next(model.parameters()).device
    preds_all = []
    targets_all = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            targets = yb.detach().cpu().numpy()
            preds_all.append(preds)
            targets_all.append(targets)
    y_pred = np.concatenate(preds_all).astype(np.int64)
    y_true = np.concatenate(targets_all).astype(np.int64)
    return y_true, y_pred


def _confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1
    return labels, cm


def _imbalanced_metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)

    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    balanced_acc = recall.mean()
    macro_f1 = f1.mean()
    overall_acc = tp.sum() / cm.sum() if cm.sum() > 0 else 0.0
    return overall_acc, balanced_acc, macro_f1, recall, precision, f1


# Evaluate on validation set
y_true_val, y_pred_val = _collect_preds_targets(model, val_dataloader)
labels, cm = _confusion_matrix(y_true_val, y_pred_val)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = (
    _imbalanced_metrics_from_cm(cm)
)

vals, cnts = np.unique(y_true_val, return_counts=True)
majority_baseline = cnts.max() / cnts.sum()

pred_vals, pred_cnts = np.unique(y_pred_val, return_counts=True)
pred_order = np.argsort(pred_cnts)[::-1]

print("Validation diagnostics")
print("-" * 60)
print(f"Majority-class baseline acc: {majority_baseline:.4f}")
print(f"Overall accuracy:            {overall_acc:.4f}")
print(f"Balanced accuracy:           {balanced_acc:.4f}")
print(f"Macro-F1:                    {macro_f1:.4f}")

print("\nPredicted class distribution (top 10):")
for cls, c in zip(pred_vals[pred_order][:10], pred_cnts[pred_order][:10]):
    print(f"  class {int(cls):3d}: {int(c)}")

print("\nPer-class metrics:")
print("class | support | recall | precision | f1")
for i, cls in enumerate(labels):
    sup = int(cm[i].sum())
    print(
        f"{int(cls):5d} | {sup:7d} | {recall[i]:6.3f} | {precision[i]:9.3f} | {f1[i]:5.3f}"
    )

print("\nConfusion matrix (rows=true, cols=pred):")
print(cm)

Validation diagnostics
------------------------------------------------------------
Majority-class baseline acc: 0.6867
Overall accuracy:            0.4007
Balanced accuracy:           0.5944
Macro-F1:                    0.3044

Predicted class distribution (top 10):
  class   4: 5689
  class   5: 4718
  class   6: 3478
  class   7: 2145
  class   3: 1684
  class   9: 1159
  class   2: 362
  class   1: 309
  class   8: 235
  class   0: 218

Per-class metrics:
class | support | recall | precision | f1
    0 |      52 |  0.808 |     0.193 | 0.311
    1 |      35 |  0.429 |     0.049 | 0.087
    2 |      71 |  0.761 |     0.149 | 0.249
    3 |     164 |  0.549 |     0.053 | 0.097
    4 |   13732 |  0.362 |     0.873 | 0.512
    5 |     724 |  0.631 |     0.097 | 0.168
    6 |    2880 |  0.354 |     0.293 | 0.321
    7 |    1446 |  0.506 |     0.341 | 0.408
    8 |      51 |  0.843 |     0.183 | 0.301
    9 |     842 |  0.702 |     0.510 | 0.591

Confusion matrix (rows=true, cols=pred):
[[

In [10]:
def majority_baseline(y, name):
    vals, cnts = np.unique(y, return_counts=True)
    p = cnts.max() / cnts.sum()
    v = vals[np.argmax(cnts)]
    print(f"{name}: n={len(y)} classes={len(vals)}")
    print(f"  majority label={v}, proportion={p:.3f}")
    print(f"  top-5:", sorted(zip(vals, cnts), key=lambda x: -x[1])[:5])


majority_baseline(y_train, "train")
majority_baseline(y_val, "val")
majority_baseline(y_test, "test")

train: n=159997 classes=10
  majority label=4, proportion=0.687
  top-5: [(4, 109856), (6, 23040), (7, 11571), (9, 6736), (5, 5795)]
val: n=19997 classes=10
  majority label=4, proportion=0.687
  top-5: [(4, 13732), (6, 2880), (7, 1446), (9, 842), (5, 724)]
test: n=20006 classes=10
  majority label=4, proportion=0.686
  top-5: [(4, 13732), (6, 2880), (7, 1447), (9, 842), (5, 725)]


## Summary: What You've Learned

This notebook covered essential concepts for training deep learning models on imbalanced remote sensing data:

### 1. **Class Imbalance Detection**
   - Analyzed training set distribution
   - Calculated imbalance ratios (e.g., majority class is 10x more frequent than rare classes)
   - Understood why simple accuracy is a poor metric for imbalanced data

### 2. **Two Strategies for Handling Imbalance**
   
   **Strategy 1: Class-Weighted Loss**
   - Pros: No resampling, clean batches, simple to implement
   - Cons: Model may still be biased toward majority class
   - Best for: Slightly imbalanced data (2-5x ratio)
   
   **Strategy 2: Weighted Random Sampler**
   - Pros: Direct control over batch composition, minorities get more training exposure
   - Cons: Resamples data, may overfit on rare classes
   - Best for: Highly imbalanced data (10x+ ratio)

### 3. **CNN Architecture for Remote Sensing**
   - Simple 2-layer CNN suitable for small patches (3×3)
   - Global average pooling instead of flattening
   - Batch normalization for stable training

### 4. **Evaluation Beyond Accuracy**
   - Balanced accuracy (mean recall per class)
   - Macro-F1 (treats all classes equally)
   - Per-class metrics (recall, precision, F1)
   - Confusion matrix analysis

---

## Comparing Your Results: Class-Weighted vs. Sampler

After training with both strategies, compare:

1. **Overall Accuracy**: May be similar, but can be misleading
2. **Balanced Accuracy**: Better indicator if classes have different sizes
3. **Macro-F1**: Fair score across all classes
4. **Per-class Recall**: Did rare classes improve?

**Typical findings:**
- Weighted loss: Good balanced accuracy, but minority classes still underperform
- Weighted sampler: Better minority performance, may overfit rare classes

---

## Advanced: Fine-Tuning Your Approach

Once you've trained and evaluated, try:

1. **Adjust class weights**: Instead of √(1/freq), try pure 1/freq or tune manually
2. **Use different sampling weights**: e.g., sample rarer classes at 2x or 3x their frequency
3. **Focal loss**: An advanced technique that focuses on hard examples
4. **Data augmentation**: Artificially increase minority class samples 
---

## Key Parameters for Your Dataset

| Parameter | Value | Notes |
|-----------|-------|-------|
| Batch Size | 64 | Smaller batches help with sampling strategies |
| Learning Rate | 3e-4 | CNN learning rate (adaptive in schedule) |
| Optimizer | AdamW | Default, includes weight decay |
| Scheduler | CosineAnnealing | Gradually reduces LR to 0 |
| Max Epochs | 50-100 | Adjust based on convergence |

---

## References & Further Reading

- **Class Imbalance**: [A Survey on Methods and Theories of Quantized Neural Networks](https://arxiv.org/abs/2106.08295)
- **Weighted Loss**: torch.nn.CrossEntropyLoss with `weight` parameter
- **Sampling**: torch.utils.data.WeightedRandomSampler
- **Evaluation**: sklearn.metrics.balanced_accuracy_score, macro_averaged F1